# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_fe_encoding.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_fe_encoding.parquet')

In [4]:
X_train.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,...,water_per_min,calories_per_step,cardio_load,cardio_efficiency,heart_step,bmi_activity,bmi_steps,bmi_calorie,sleep_stress_interaction,water_to_calorie_ratio
id,,,,,,,,,,,,,,,,,,,,,
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,0.226287,0.560855,0.224078,...,0.089423,1.638282,1397.88,30.363128,0.053203,1.233654,0.019337,0.011798,0.557395,0.000855
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,0.212810,0.206102,0.224078,...,0.024754,0.198746,3557.87,27.192254,0.007208,0.507662,0.002612,0.013137,0.204425,0.000641
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,0.226287,0.560855,0.306270,...,0.040921,0.189069,2872.74,35.183246,0.005304,0.627621,0.001726,0.009126,0.868247,0.000595
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,0.226287,0.560855,0.224078,...,0.033169,0.366551,4624.28,33.631714,0.010760,0.379803,0.003224,0.008791,0.557395,0.000768
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,0.226287,0.224644,0.224078,...,0.047872,0.388762,3376.40,34.408602,0.011147,0.605106,0.004319,0.011105,0.222635,0.000879


In [5]:
X_test.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,...,water_per_min,calories_per_step,cardio_load,cardio_efficiency,heart_step,bmi_activity,bmi_steps,bmi_calorie,sleep_stress_interaction,water_to_calorie_ratio
id,,,,,,,,,,,,,,,,,,,,,
690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,0.226287,0.560855,0.306270,...,0.030744,0.193746,3861.55,41.654021,0.004581,0.388099,0.001657,0.008551,0.868247,0.000677
690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,0.236020,0.560855,0.306270,...,0.094118,0.260659,2035.95,21.082045,0.012217,0.879216,0.003296,0.012638,0.868247,0.001353
690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,0.236020,0.009132,0.306270,...,0.055758,0.229417,2895.45,50.082372,0.004505,0.487677,0.001822,0.007938,0.008623,0.000908
690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,0.226287,0.206102,0.142713,...,0.040415,0.393872,4466.65,31.371069,0.012397,0.453541,0.004147,0.010525,0.282506,0.000938
690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,0.226287,0.560855,0.224078,...,0.060644,0.131558,3061.38,23.227446,0.005592,0.576485,0.001676,0.012734,0.557395,0.001340


In [6]:
cat_features_raw = X_train.select_dtypes(include=['category', 'object']).columns.tolist()

X_train = X_train.astype({feat: 'category' for feat in cat_features_raw})
X_test = X_test.astype({feat: 'category' for feat in cat_features_raw})

cat_features_raw

[]

# Machine Learning

In [7]:
models = dict(
    lgbm=load_pickle('../models/layer_1/lightgbm_n_trial_60_data_type_feat_eng_target_encoding.pkl'),
    xgb=load_pickle('../models/layer_1/xgboost_n_trial_60_data_type_feat_eng_target_encoding.pkl'),
    # cat=load_pickle('../models/layer_1/catboost_n_trial_60_data_type_raw.pkl'),
)

## Train Dataset

In [8]:
X_train_stacking = pd.DataFrame({})

In [9]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(
        model, 
        X_train, 
        y_train.health_condition,
        method='predict_proba', 
        cv=StratifiedKFold(shuffle=True, random_state=42, n_splits=3),
        params={'cat_features': cat_features_raw} if model_name == 'cat' else None,
        n_jobs=1 if model_name == 'cat' else 1
    )
    
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                        | 0/2 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 50%|███████████████████████████████████████████████████████████████████████████████████████▌                                                                                       | 1/2 [02:51<02:51, 171.47s/it]

Predicting Train Dataset xgb


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [06:13<00:00, 186.98s/it]


## Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({})

In [11]:
for model_name, model in models.items():
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

Predicting Test Dataset lgbm
Predicting Test Dataset xgb


# Saving

In [12]:
X_train_stacking.to_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials_data_type_feat_eng_taret_encoding.parquet')
X_test_stacking.to_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials_data_type_feat_eng_target_encoding.parquet')

In [13]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.001638,0.000283,0.998079,0.001796,0.000363,0.997841
1,0.996015,0.000339,0.003646,0.997132,0.000288,0.002580
2,0.009099,0.001175,0.989726,0.005254,0.000682,0.994064
3,0.003855,0.002145,0.994000,0.004080,0.001686,0.994234
4,0.995730,0.000018,0.004252,0.993613,0.000010,0.006377


In [14]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.013297,0.004871,0.981832,0.011730,0.004806,0.983465
1,0.407736,0.001829,0.590435,0.492656,0.003306,0.504038
2,0.998801,0.001053,0.000147,0.998502,0.001166,0.000333
3,0.976600,0.000013,0.023387,0.988312,0.000014,0.011675
4,0.004873,0.002234,0.992892,0.007801,0.002434,0.989766


In [15]:
X_train.shape

(690088, 25)

In [16]:
X_test.shape

(295753, 25)

In [17]:
y_train.head()

,health_condition
id,
0,2
1,0
2,2
3,2
4,0
